# Phase 4 Region and Hybrid Line/Word Segmentation Colab Workflow

This notebook implements the image-segmentation half of the thesis pipeline in Colab/Jupyter.

## Final recommended approach

1. Preprocess full prescription images from `data/raw`.
2. Train or reuse the YOLO handwritten-region detector.
3. Crop handwritten prescription regions with `models/region_yolo_best.pt`.
4. Segment lines inside each region using the hybrid OpenCV line segmenter.
5. Segment words from each line crop for word-level TrOCR.

The current region YOLO result is already strong, so raw/full-page augmentation is optional and not required by default. The YOLO line detector remains in this notebook as an optional comparison experiment, but the final pipeline should use region YOLO + hybrid line/word segmentation because the line dataset is small.


## Step 0: Runtime Notes

Use a GPU runtime in Colab when training YOLO. The annotation cells use `ipywidgets`, so run them in order and save after each box. If you are using Google Drive, mount it and set `REPO_DIR` to your project folder.

In [ ]:
# Colab/Drive project setup.
# This cell mounts Google Drive, keeps the repository in Drive, and keeps data in Drive.
# Put raw prescriptions here after this cell runs:
# /content/drive/MyDrive/phase4_project/repo/data/raw

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    if not (REPO_DIR / 'pipeline').exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(
                f'{REPO_DIR} exists but does not look like the project repo. '
                'Move/rename it or set REPO_DIR to the correct folder.'
            )
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    # Local Jupyter/VS Code fallback: use the current repository folder.
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)

# data/ is gitignored, so it is safe to keep raw images and generated files here.
# In Colab this folder is inside Google Drive and persists across sessions.
DATA_DIR = REPO_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('Repository:', REPO_DIR)
print('Data folder:', DATA_DIR)
print('Current working directory:', Path.cwd())
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())
print('Raw data folder:', RAW_DIR)
print('Raw images currently:', len([p for p in RAW_DIR.glob('*') if p.is_file()]))

## Step 1: Install Dependencies

This installs the image-processing dependencies and Ultralytics YOLO. OCR dependencies are not required for the first two pipeline stages.

In [ ]:
# Install dependencies for preprocessing, annotation manifests, OpenCV segmentation, and YOLO training.
!python3 -m pip install -q -r pipeline/requirements.txt
!python3 -m pip install -q -r pipeline/requirements-layout.txt ipywidgets

## Step 2: Define Paths

All generated files stay under `data/processed_region_line` so you can rerun this notebook without overwriting the OCR dataset.

In [ ]:
from pathlib import Path

RAW_DIR = Path('data/raw')
WORK_DIR = Path('data/processed_region_line')
PAGES_DIR = WORK_DIR / 'pages'
PAGE_MANIFEST = WORK_DIR / 'page_manifest.csv'

REGIONS_DIR = WORK_DIR / 'regions'
REGION_MANIFEST = WORK_DIR / 'region_manifest.csv'
REGION_YOLO_DATASET = Path('data/yolo_region_dataset')
MODEL_DIR = Path('models')
REGION_MODEL = MODEL_DIR / 'region_yolo_best.pt'

LINE_CROPS_DIR = WORK_DIR / 'line_crops'
LINE_MANIFEST = WORK_DIR / 'line_manifest.csv'
CORRECTED_LINE_MANIFEST = WORK_DIR / 'line_manifest_corrected.csv'
LINE_YOLO_DATASET = Path('data/yolo_line_dataset')
LINE_MODEL = MODEL_DIR / 'line_yolo_best.pt'

for p in [WORK_DIR, PAGES_DIR, REGIONS_DIR, LINE_CROPS_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Raw images:', RAW_DIR)
print('Work dir:', WORK_DIR)

## Step 3: Preprocess Full Prescription Pages

This stage resizes large images, estimates page skew, rotates the page, denoises it, and improves contrast. The output is a processed page image plus `page_manifest.csv`.

In [ ]:
# Preprocess all full prescription images from data/raw.
!python3 pipeline/scripts/preprocess_pages.py \
  --input-dir "{RAW_DIR}" \
  --output-dir "{PAGES_DIR}" \
  --manifest-out "{PAGE_MANIFEST}" \
  --max-side 2200

import pandas as pd
page_df = pd.read_csv(PAGE_MANIFEST)
page_df.head()

## Step 4: Preview Processed Pages

Use this to confirm that the raw images loaded correctly and that page deskewing did not damage the prescription.

In [ ]:
from IPython.display import display, Image as IPImage

sample_pages = sorted(PAGES_DIR.glob('*.jpg'))[:5]
print('Preview pages:', len(sample_pages))
for p in sample_pages:
    print(p)
    display(IPImage(filename=str(p), width=650))

# Stage 1: Region Segmentation

The goal of this stage is to remove printed header/footer content and isolate the handwritten prescription region. There are two annotation options:

Use the in-notebook manual cropper for a quick project workflow, or use CVAT/Label Studio if you prefer a full annotation tool. After corrected region boxes exist, train YOLO so the region proposal becomes learned rather than heuristic.

## Step 5A: Prepare CVAT/Label Studio Package for Region Annotation

Annotate the processed pages with these classes in this order: `header`, `handwritten_region`, `footer`. Export YOLO labels if you use CVAT/Label Studio. The final cropper will use only `handwritten_region`.

In [ ]:
# Creates image package and class file for CVAT/Label Studio region annotation.
!python3 pipeline/scripts/prepare_layout_annotation.py \
  --pages-dir "{PAGES_DIR}" \
  --output-dir "{WORK_DIR / 'layout_annotation_package'}" \
  --classes header handwritten_region footer \
  --copy-images

print('Upload this folder to CVAT/Label Studio if using external annotation:')
print(WORK_DIR / 'layout_annotation_package')
print('After export, put YOLO .txt labels in:', WORK_DIR / 'layout_yolo_labels')

## Step 5B: Manual Region Cropper Inside Notebook

Use this when you want to avoid CVAT. Move the sliders until the green box contains only the handwritten prescription area. Click **Save Region**. Each saved crop is appended to `region_manifest.csv`.

In [ ]:
# Manual handwritten-region cropper.
# The saved CSV is the training annotation file for the YOLO region detector.
import csv
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage

manual_pages = sorted(PAGES_DIR.glob('*.jpg'))
if not manual_pages:
    raise FileNotFoundError(f'No processed pages found in {PAGES_DIR}. Run preprocessing first.')

REGIONS_DIR.mkdir(parents=True, exist_ok=True)
manual_state = {'page_idx': 0, 'region_idx': 0}
manual_out = widgets.Output()
preview_out = widgets.Output()
page_label = widgets.Label()

x1_slider = widgets.IntSlider(description='x1', min=0, max=100, value=0, continuous_update=False)
y1_slider = widgets.IntSlider(description='y1', min=0, max=100, value=0, continuous_update=False)
x2_slider = widgets.IntSlider(description='x2', min=1, max=100, value=100, continuous_update=False)
y2_slider = widgets.IntSlider(description='y2', min=1, max=100, value=100, continuous_update=False)
prev_btn = widgets.Button(description='Previous Page')
next_btn = widgets.Button(description='Next Page')
preview_btn = widgets.Button(description='Preview Crop')
save_btn = widgets.Button(description='Save Region', button_style='success')


def current_page_path():
    return manual_pages[manual_state['page_idx']]


def update_sliders():
    image = cv2.imread(str(current_page_path()))
    h, w = image.shape[:2]
    x1_slider.max, x2_slider.max = w - 1, w
    y1_slider.max, y2_slider.max = h - 1, h
    x1_slider.value = 0
    y1_slider.value = int(h * 0.15)
    x2_slider.value = w
    y2_slider.value = int(h * 0.90)
    page_label.value = f'Page {manual_state["page_idx"] + 1}/{len(manual_pages)}: {current_page_path().name} ({w}x{h})'
    show_preview(full_page=True)


def get_box():
    image = cv2.imread(str(current_page_path()))
    h, w = image.shape[:2]
    x1 = max(0, min(x1_slider.value, w - 1))
    y1 = max(0, min(y1_slider.value, h - 1))
    x2 = max(x1 + 1, min(x2_slider.value, w))
    y2 = max(y1 + 1, min(y2_slider.value, h))
    return image, x1, y1, x2, y2


def show_preview(full_page=False):
    image, x1, y1, x2, y2 = get_box()
    vis = image.copy()
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 220, 0), 4)
    tmp_path = WORK_DIR / '_manual_region_preview.jpg'
    cv2.imwrite(str(tmp_path), vis if full_page else image[y1:y2, x1:x2])
    with preview_out:
        clear_output(wait=True)
        display(IPImage(filename=str(tmp_path), width=700))


def append_region(row):
    columns = ['region_id', 'page_id', 'region_label', 'page_image_path', 'region_image_path', 'x1_page', 'y1_page', 'x2_page', 'y2_page', 'width', 'height']
    exists = REGION_MANIFEST.exists()
    with REGION_MANIFEST.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def save_region(_):
    image, x1, y1, x2, y2 = get_box()
    page_path = current_page_path()
    page_id = page_path.stem
    region_id = f'{page_id}_manual_r{manual_state["region_idx"]:02d}'
    region_path = REGIONS_DIR / f'{region_id}.png'
    cv2.imwrite(str(region_path), image[y1:y2, x1:x2])
    append_region({
        'region_id': region_id,
        'page_id': page_id,
        'region_label': 'handwritten_region',
        'page_image_path': str(page_path),
        'region_image_path': str(region_path),
        'x1_page': x1,
        'y1_page': y1,
        'x2_page': x2,
        'y2_page': y2,
        'width': x2 - x1,
        'height': y2 - y1,
    })
    manual_state['region_idx'] += 1
    with manual_out:
        clear_output(wait=True)
        print('Saved region:', region_path)
        print('Updated manifest:', REGION_MANIFEST)


def prev_page(_):
    manual_state['page_idx'] = max(0, manual_state['page_idx'] - 1)
    update_sliders()


def next_page(_):
    manual_state['page_idx'] = min(len(manual_pages) - 1, manual_state['page_idx'] + 1)
    update_sliders()

prev_btn.on_click(prev_page)
next_btn.on_click(next_page)
preview_btn.on_click(lambda _: show_preview(full_page=False))
save_btn.on_click(save_region)

ui = widgets.VBox([
    page_label,
    widgets.HBox([prev_btn, next_btn, preview_btn, save_btn]),
    widgets.HBox([x1_slider, x2_slider]),
    widgets.HBox([y1_slider, y2_slider]),
    preview_out,
    manual_out,
])

display(ui)
update_sliders()

## Step 6: Train YOLO Region Detector

This converts corrected handwritten-region boxes into a YOLO dataset with train, validation, and test splits. With 53 full prescription images, a practical split is about 70/15/15, which gives roughly 37 training pages, 8 validation pages, and 8 test pages.

YOLO already performs online augmentation during training. This is better than permanently duplicating many augmented files because each epoch sees different rotations, scale changes, shear, and perspective changes while the test set remains clean and honest.


In [ ]:
from pathlib import Path
REGION_MODEL = Path('models/region_yolo_best.pt')
REGION_MODEL.parent.mkdir(parents=True, exist_ok=True)

# Convert region_manifest.csv to a one-class YOLO dataset: handwritten_region.
# For 53 prescriptions, this creates an approximate 70/15/15 train/val/test split.
!python3 pipeline/scripts/prepare_yolo_layout_dataset.py \
  --page-manifest "{PAGE_MANIFEST}" \
  --region-manifest "{REGION_MANIFEST}" \
  --output-dir "{REGION_YOLO_DATASET}" \
  --class-name handwritten_region \
  --val-ratio 0.15 \
  --test-ratio 0.15 \
  --seed 42

# Train YOLO with online augmentation for skew, camera perspective, and scale variation.
# If T4 memory is low, change --imgsz 960 to 640 and --batch 8 to 4.
!python3 pipeline/scripts/train_yolo_layout.py \
  --data-yaml "{REGION_YOLO_DATASET / 'data.yaml'}" \
  --model yolov8n.pt \
  --epochs 80 \
  --imgsz 960 \
  --batch 8 \
  --project runs/layout \
  --name handwritten_region_yolo \
  --degrees 5 \
  --shear 1.5 \
  --perspective 0.0005 \
  --scale 0.35 \
  --patience 20 \
  --weights-out "{REGION_MODEL}"

# If this notebook was run before the stable copy feature, recover the latest best.pt.
if not REGION_MODEL.exists():
    candidates = sorted(Path('.').glob('**/handwritten_region_yolo/weights/best.pt'), key=lambda x: x.stat().st_mtime)
    if candidates:
        import shutil
        REGION_MODEL.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidates[-1], REGION_MODEL)
        print('Recovered region weights from:', candidates[-1])

REGION_MODEL = Path('models/region_yolo_best.pt')
print('Using region model:', REGION_MODEL.resolve(), 'exists=', REGION_MODEL.exists())

# Evaluate on the held-out test split.
!yolo detect val \
  model="{REGION_MODEL}" \
  data="{REGION_YOLO_DATASET / 'data.yaml'}" \
  split=test \
  imgsz=960

print('Best region model:', REGION_MODEL)


## Step 7: Run Region Model to Generate Clean Region Crops

After training, run the first stage with the YOLO region model. The output `region_manifest.csv` contains the learned region proposal boxes.

In [ ]:
from pathlib import Path
REGION_MODEL = Path('models/region_yolo_best.pt')
print('Using region model:', REGION_MODEL.resolve(), REGION_MODEL.exists())

# Run region proposal with trained YOLO and stop before OCR.
# target-class is 0 because the trained model has one class: handwritten_region.
!python3 pipeline/scripts/run_end_to_end.py \
  --input "{RAW_DIR}" \
  --output-dir "{WORK_DIR / 'region_model_output'}" \
  --yolo-model "{REGION_MODEL}" \
  --target-class 0 \
  --ocr-backend none \
  --ocr-unit word

print('Learned region manifest:', WORK_DIR / 'region_model_output' / 'region_manifest.csv')
print('Hybrid line manifest:', WORK_DIR / 'region_model_output' / 'line_manifest.csv')
print('Hybrid word manifest:', WORK_DIR / 'region_model_output' / 'word_manifest.csv')

# Stage 2: Line Segmentation

The input to this stage is each handwritten region crop. The baseline script uses OpenCV thresholding and horizontal projection. This works for clean horizontal writing, but it can fail on slanted, curved, overlapping, or widely spaced text. For final evaluation, annotate/correct line boxes and train a YOLO line detector.

## Step 8A: Bootstrap Line Crops With Heuristic Segmentation

This creates initial line boxes quickly. Use it as a starting point, then correct bad boxes manually.

In [ ]:
# Use the corrected/manual region manifest unless you want to use region_model_output.
REGION_MANIFEST_FOR_LINES = REGION_MANIFEST
# REGION_MANIFEST_FOR_LINES = WORK_DIR / 'region_model_output' / 'region_manifest.csv'

!python3 pipeline/scripts/segment_lines.py \
  --region-manifest "{REGION_MANIFEST_FOR_LINES}" \
  --output-dir "{LINE_CROPS_DIR}" \
  --manifest-out "{LINE_MANIFEST}" \
  --min-line-height 14 \
  --merge-gap 8 \
  --line-padding 6

import pandas as pd
line_df = pd.read_csv(LINE_MANIFEST)
print('Lines:', len(line_df))
line_df.head()

## Step 8B: Manual Line Box Annotator Inside Notebook

Use this to correct line segmentation. The saved file is `line_manifest_corrected.csv`, which is used to train the YOLO line detector. A line box should contain one prescription/instruction line, even if it is slightly slanted. Keep enough vertical padding so the OCR model does not lose ascenders/descenders.

For regions that contain multiple lines, annotate them one by one inside the same region. Adjust the box around the first line and click `Save Line`; the box will move downward automatically for the next line. Adjust it again if needed and click `Save Line` again. Use `Next Line Same Region` only to move the box downward without saving. Click `Next Region` only after every line in the current region has been saved.


In [ ]:
# Manual line annotator for region crops.
# It saves corrected line boxes relative to each region image.
import csv
import cv2
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage

regions_df = pd.read_csv(REGION_MANIFEST_FOR_LINES)
if regions_df.empty:
    raise ValueError('No regions available for line annotation.')

line_state = {'region_idx': 0, 'line_idx': 0, 'refresh_enabled': True}
line_preview_out = widgets.Output()
line_save_out = widgets.Output()
region_label = widgets.Label()

lx1 = widgets.IntSlider(description='x1', min=0, max=100, value=0, continuous_update=True)
ly1 = widgets.IntSlider(description='y1', min=0, max=100, value=0, continuous_update=True)
lx2 = widgets.IntSlider(description='x2', min=1, max=100, value=100, continuous_update=True)
ly2 = widgets.IntSlider(description='y2', min=1, max=100, value=100, continuous_update=True)
prev_region_btn = widgets.Button(description='Previous Region')
next_region_btn = widgets.Button(description='Next Region')
preview_line_btn = widgets.Button(description='Preview Crop')
save_line_btn = widgets.Button(description='Save Line', button_style='success')
next_line_btn = widgets.Button(description='Next Line Same Region')


def current_region_row():
    return regions_df.iloc[line_state['region_idx']]


def current_region_image():
    row = current_region_row()
    image = cv2.imread(str(row['region_image_path']))
    if image is None:
        raise FileNotFoundError(row['region_image_path'])
    return image


def saved_line_count_for_region(region_id):
    if not CORRECTED_LINE_MANIFEST.exists():
        return 0
    try:
        manifest_df = pd.read_csv(CORRECTED_LINE_MANIFEST)
    except pd.errors.EmptyDataError:
        return 0
    if 'region_id' not in manifest_df.columns:
        return 0
    return int((manifest_df['region_id'].astype(str) == str(region_id)).sum())


def update_region_label():
    image = current_region_image()
    h, w = image.shape[:2]
    row = current_region_row()
    region_label.value = (
        f'Region {line_state["region_idx"] + 1}/{len(regions_df)}: '
        f'{row["region_id"]} ({w}x{h}) | saved lines here: {line_state["line_idx"]}'
    )


def update_line_sliders():
    image = current_region_image()
    h, w = image.shape[:2]
    line_state['refresh_enabled'] = False
    lx1.max, lx2.max = w - 1, w
    ly1.max, ly2.max = h - 1, h
    lx1.value = 0
    ly1.value = int(h * 0.05)
    lx2.value = w
    ly2.value = min(h, int(h * 0.18))
    row = current_region_row()
    line_state['line_idx'] = saved_line_count_for_region(row['region_id'])
    line_state['refresh_enabled'] = True
    update_region_label()
    show_line_preview(full_region=True)


def get_line_box():
    image = current_region_image()
    h, w = image.shape[:2]
    x1 = max(0, min(lx1.value, w - 1))
    y1 = max(0, min(ly1.value, h - 1))
    x2 = max(x1 + 1, min(lx2.value, w))
    y2 = max(y1 + 1, min(ly2.value, h))
    return image, x1, y1, x2, y2


def display_bgr_image(image, width=700):
    ok, encoded = cv2.imencode('.jpg', image)
    if not ok:
        raise RuntimeError('Could not encode preview image.')
    display(IPImage(data=encoded.tobytes(), width=width))


def show_line_preview(full_region=True):
    image, x1, y1, x2, y2 = get_line_box()
    if full_region:
        vis = image.copy()
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 220, 0), 3)
        preview = vis
    else:
        preview = image[y1:y2, x1:x2]
    with line_preview_out:
        clear_output(wait=True)
        display_bgr_image(preview, width=700)


def refresh_preview_on_slider_change(change):
    if line_state.get('refresh_enabled', True) and change['name'] == 'value':
        show_line_preview(full_region=True)


def move_to_next_line(_=None):
    image = current_region_image()
    h, _ = image.shape[:2]
    current_height = max(20, ly2.value - ly1.value)
    gap = max(8, int(current_height * 0.30))
    new_y1 = ly2.value + gap
    if new_y1 >= h - 1:
        new_y1 = max(0, h - current_height)
    new_y2 = min(h, new_y1 + current_height)
    line_state['refresh_enabled'] = False
    ly1.value = max(0, min(new_y1, h - 1))
    ly2.value = max(ly1.value + 1, new_y2)
    line_state['refresh_enabled'] = True
    show_line_preview(full_region=True)


def append_line(row):
    columns = [
        'line_id', 'page_id', 'region_id', 'line_image_path', 'region_image_path',
        'page_image_path', 'x1_region', 'y1_region', 'x2_region', 'y2_region',
        'x1_page', 'y1_page', 'x2_page', 'y2_page'
    ]
    exists = CORRECTED_LINE_MANIFEST.exists()
    with CORRECTED_LINE_MANIFEST.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def save_line(_):
    image, x1, y1, x2, y2 = get_line_box()
    row = current_region_row()
    corrected_dir = WORK_DIR / 'corrected_line_crops'
    corrected_dir.mkdir(parents=True, exist_ok=True)
    line_id = f'{row["region_id"]}_manual_l{line_state["line_idx"]:02d}'
    line_path = corrected_dir / f'{line_id}.png'
    cv2.imwrite(str(line_path), image[y1:y2, x1:x2])
    rx1 = int(row.get('x1_page', 0))
    ry1 = int(row.get('y1_page', 0))
    append_line({
        'line_id': line_id,
        'page_id': row['page_id'],
        'region_id': row['region_id'],
        'line_image_path': str(line_path),
        'region_image_path': row['region_image_path'],
        'page_image_path': row.get('page_image_path', ''),
        'x1_region': x1,
        'y1_region': y1,
        'x2_region': x2,
        'y2_region': y2,
        'x1_page': rx1 + x1,
        'y1_page': ry1 + y1,
        'x2_page': rx1 + x2,
        'y2_page': ry1 + y2,
    })
    line_state['line_idx'] += 1
    update_region_label()
    with line_save_out:
        clear_output(wait=True)
        print('Saved line:', line_path)
        print('Saved lines in this region:', line_state['line_idx'])
        print('Updated manifest:', CORRECTED_LINE_MANIFEST)
    move_to_next_line()


def prev_region(_):
    line_state['region_idx'] = max(0, line_state['region_idx'] - 1)
    update_line_sliders()


def next_region(_):
    line_state['region_idx'] = min(len(regions_df) - 1, line_state['region_idx'] + 1)
    update_line_sliders()

for slider in [lx1, ly1, lx2, ly2]:
    slider.observe(refresh_preview_on_slider_change, names='value')

prev_region_btn.on_click(prev_region)
next_region_btn.on_click(next_region)
preview_line_btn.on_click(lambda _: show_line_preview(full_region=False))
save_line_btn.on_click(save_line)
next_line_btn.on_click(move_to_next_line)

line_ui = widgets.VBox([
    region_label,
    widgets.HBox([prev_region_btn, next_region_btn]),
    widgets.HBox([preview_line_btn, save_line_btn, next_line_btn]),
    widgets.HBox([lx1, lx2]),
    widgets.HBox([ly1, ly2]),
    line_preview_out,
    line_save_out,
])

display(line_ui)
update_line_sliders()


## Step 9: Optional YOLO Line Detector Experiment

This converts corrected line boxes into a one-class YOLO dataset. It is useful for comparison in the thesis, but it is not the recommended final line segmenter for the current data size.

Your previous line YOLO result was weak (`mAP50-95` around 0.09), while the hybrid OpenCV line segmenter works directly on the good region crops. Keep this cell optional unless you collect many more line annotations.


In [ ]:
from pathlib import Path
LINE_MODEL = Path('models/line_yolo_best.pt')
LINE_MODEL.parent.mkdir(parents=True, exist_ok=True)

# Convert corrected line boxes into YOLO format.
# Final training should use CORRECTED_LINE_MANIFEST.
# If you are only smoke-testing, this falls back to the heuristic LINE_MANIFEST.
LINE_TRAIN_MANIFEST = CORRECTED_LINE_MANIFEST if CORRECTED_LINE_MANIFEST.exists() else LINE_MANIFEST

!python3 pipeline/scripts/prepare_yolo_line_dataset.py \
  --line-manifest "{LINE_TRAIN_MANIFEST}" \
  --output-dir "{LINE_YOLO_DATASET}" \
  --class-name handwritten_line \
  --val-ratio 0.15 \
  --test-ratio 0.15 \
  --seed 42

# Train YOLO line detector with conservative augmentation for thin line boxes.
# If T4 memory is low, use --imgsz 960 and --batch 4.
!python3 pipeline/scripts/train_yolo_line.py \
  --data-yaml "{LINE_YOLO_DATASET / 'data.yaml'}" \
  --model yolov8n.pt \
  --epochs 150 \
  --imgsz 1280 \
  --batch 4 \
  --project runs/lines \
  --name handwritten_line_yolo \
  --degrees 2 \
  --shear 0 \
  --perspective 0.0 \
  --scale 0.15 \
  --mosaic 0 \
  --mixup 0 \
  --copy-paste 0 \
  --close-mosaic 0 \
  --patience 35 \
  --weights-out "{LINE_MODEL}"

# If this notebook was run before the stable copy feature, recover the latest best.pt.
if not LINE_MODEL.exists():
    candidates = sorted(Path('.').glob('**/handwritten_line_yolo/weights/best.pt'), key=lambda x: x.stat().st_mtime)
    if candidates:
        import shutil
        LINE_MODEL.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidates[-1], LINE_MODEL)
        print('Recovered line weights from:', candidates[-1])

LINE_MODEL = Path('models/line_yolo_best.pt')
print('Using line model:', LINE_MODEL.resolve(), 'exists=', LINE_MODEL.exists())

# Evaluate on the held-out test split.
!yolo detect val \
  model="{LINE_MODEL}" \
  data="{LINE_YOLO_DATASET / 'data.yaml'}" \
  split=test \
  imgsz=1280

print('Best line model:', LINE_MODEL)


## Step 10: Run Final Segmentation Pipeline

This is the recommended thesis-order execution for segmentation: full prescription image to handwritten region crops, then hybrid line crops, then word crops. It uses the trained YOLO model only for handwritten-region detection and does not use the YOLO line detector.

The generated `word_manifest.csv` is the bridge into the TrOCR notebook and the annotation notebook.


In [ ]:
from pathlib import Path
REGION_MODEL = Path('models/region_yolo_best.pt')
FINAL_SEGMENTATION_DIR = Path('data/final_region_hybrid_line_word')
print('Region model:', REGION_MODEL.resolve(), REGION_MODEL.exists())

# Recommended final run: trained region YOLO + hybrid OpenCV line segmentation + word crops, no OCR.
!python3 pipeline/scripts/run_end_to_end.py \
  --input "{RAW_DIR}" \
  --output-dir "{FINAL_SEGMENTATION_DIR}" \
  --yolo-model "{REGION_MODEL}" \
  --target-class 0 \
  --ocr-backend none \
  --ocr-unit word \
  --line-padding 6

print('Final segmentation outputs:')
print('Region manifest:', FINAL_SEGMENTATION_DIR / 'region_manifest.csv')
print('Line manifest:', FINAL_SEGMENTATION_DIR / 'line_manifest.csv')
print('Word manifest:', FINAL_SEGMENTATION_DIR / 'word_manifest.csv')
print('Line crop folder:', FINAL_SEGMENTATION_DIR / 'line_crops' / 'lines')
print('Word crop folder:', FINAL_SEGMENTATION_DIR / 'line_crops' / 'words')


## Practical Decisions

### Should raw/full-page augmentation be used for handwritten-region YOLO?

Not by default. Your current region detector is already strong, so keep the region model and only retrain with extra raw augmentation if it fails on new forms, new camera angles, or different lighting.

### Where should augmentation happen now?

Use train-only augmentation for the word-level OCR dataset. Small rotate/shift/scale/brightness/noise changes help TrOCR generalize without corrupting the validation or test measurements.

### What handles skew?

Page skew is handled in preprocessing. Slanted handwriting is handled by generous line padding and word-level crops. Axis-aligned crops are acceptable for OCR as long as the full text stroke remains inside the crop.
